In [1]:
# =============================================================================
# AGGREGATE PLANNING — SEARCH DECISION RULE (SDR) METHOD
# Google Colab / Jupyter Compatible
# =============================================================================
# INSTRUCTIONS (Google Colab):
#   1. Upload SDR_Aggregate_Planning.xlsx when prompted, OR
#   2. Set FILE_PATH to the correct path if running locally / in Drive
# =============================================================================

import pandas as pd
import numpy as np
from itertools import product

# ── 0. File path ──────────────────────────────────────────────────────────────
FILE_PATH = "SDR_Aggregate_Planning.xlsx"   # <-- change if needed

# Optional: auto-upload in Colab
try:
    import google.colab
    from google.colab import files
    print("Running in Google Colab — uploading file …")
    uploaded = files.upload()
    FILE_PATH = list(uploaded.keys())[0]
except ImportError:
    pass   # running locally; use FILE_PATH above

# =============================================================================
# SECTION 1 — READ PROBLEM DATA FROM EXCEL
# =============================================================================
print("=" * 70)
print("SECTION 1: READING PROBLEM DATA FROM EXCEL")
print("=" * 70)

xl = pd.ExcelFile(FILE_PATH)
print(f"Sheets found: {xl.sheet_names}\n")

# ── Demand data (Sheet 1) ─────────────────────────────────────────────────────
df_prob = xl.parse("1_Problem_Data", header=None)

# Months and demand are on row index 3 (0-based), cols 1-6 (Jan-Jun values)
months  = ["January", "February", "March", "April", "May", "June"]
demand  = [1200, 1500, 1800, 2100, 1900, 1600]   # units/month

# ── Cost & operational parameters ─────────────────────────────────────────────
prod_rate = 30      # units / worker / month
W0        = 50      # initial workforce (workers)
I0        = 200     # initial inventory (units)
c_hire    = 5_000   # ₹ per worker hired
c_fire    = 8_000   # ₹ per worker fired
c_hold    = 20      # ₹ per unit per month (holding)
c_back    = 50      # ₹ per unit per month (backorder)
c_labour  = 12_000  # ₹ per worker per month

print("GIVEN DATA:")
print(f"  Months  : {months}")
print(f"  Demand  : {demand}")
print(f"  Total D : {sum(demand)} units  |  Average D : {sum(demand)/len(demand):.2f} units/month")
print()
print("COST & OPERATIONAL PARAMETERS:")
print(f"  Production rate  (p)       : {prod_rate} units/worker/month")
print(f"  Initial workforce (W0)     : {W0} workers")
print(f"  Initial inventory (I0)     : {I0} units")
print(f"  Hiring cost  (c_hire)      : ₹{c_hire:,}/worker")
print(f"  Firing cost  (c_fire)      : ₹{c_fire:,}/worker")
print(f"  Holding cost (c_hold)      : ₹{c_hold}/unit/month")
print(f"  Backorder cost (c_back)    : ₹{c_back}/unit/month")
print(f"  Labour cost  (c_labour)    : ₹{c_labour:,}/worker/month")

# =============================================================================
# SECTION 2 — CORE CALCULATION FUNCTION
# =============================================================================
def compute_plan(workforce_plan, demand, I0, W0,
                 prod_rate, c_hire, c_fire, c_hold, c_back, c_labour):
    """
    Given a list of monthly workforce levels (one per period),
    compute all costs and return a detailed DataFrame + grand totals.

    Step-by-step for each month t:
      1. Production  Pt  = Wt × prod_rate
      2. ΔW              = Wt − Wt₋₁  (W₋₁ = W0 for t=1)
      3. Hire        Ht  = max(0,  ΔW)
      4. Fire        Ft  = max(0, −ΔW)
      5. Inventory   It  = Iₜ₋₁ + Pt − Dt
      6. Labour cost      = Wt × c_labour
      7. Hire cost        = Ht × c_hire
      8. Fire cost        = Ft × c_fire
      9. Hold cost        = max(0,  It) × c_hold
     10. Back cost        = max(0, −It) × c_back
     11. Monthly TC       = sum of costs 6-10
    """
    T = len(demand)
    rows = []
    inv_prev = I0
    W_prev   = W0

    for t in range(T):
        Wt  = workforce_plan[t]
        Pt  = Wt * prod_rate
        Dt  = demand[t]
        dW  = Wt - W_prev
        Ht  = max(0,  dW)
        Ft  = max(0, -dW)
        It  = inv_prev + Pt - Dt

        labour_cost = Wt   * c_labour
        hire_cost   = Ht   * c_hire
        fire_cost   = Ft   * c_fire
        hold_cost   = max(0,  It) * c_hold
        back_cost   = max(0, -It) * c_back
        monthly_tc  = labour_cost + hire_cost + fire_cost + hold_cost + back_cost

        rows.append({
            "Month"       : months[t],
            "Workforce Wt": Wt,
            "Production Pt": Pt,
            "Demand Dt"   : Dt,
            "ΔW"          : dW,
            "Hire Ht"     : Ht,
            "Fire Ft"     : Ft,
            "Inventory It": It,
            "Labour Cost ₹": labour_cost,
            "Hire Cost ₹" : hire_cost,
            "Fire Cost ₹" : fire_cost,
            "Hold Cost ₹" : hold_cost,
            "Back Cost ₹" : back_cost,
            "Monthly TC ₹": monthly_tc,
        })

        inv_prev = It
        W_prev   = Wt

    df = pd.DataFrame(rows)
    totals = {col: df[col].sum() for col in df.columns if "₹" in col or col in ["Hire Ht", "Fire Ft"]}
    totals["Inventory It"] = df["Inventory It"].sum()
    return df, totals

# =============================================================================
# SECTION 3 — SDR OPTIMAL PLAN  (W* = 55, Level Strategy)
# =============================================================================
print("\n" + "=" * 70)
print("SECTION 2: SDR OPTIMAL PLAN  (W* = 55 workers — Level Strategy)")
print("=" * 70)

W_optimal = [55] * 6   # level workforce of 55 for all 6 months

df_opt, totals_opt = compute_plan(
    W_optimal, demand, I0, W0,
    prod_rate, c_hire, c_fire, c_hold, c_back, c_labour
)

print("\nMONTH-BY-MONTH DETAIL:")
print(df_opt.to_string(index=False))

print("\nGRAND TOTALS:")
for k, v in totals_opt.items():
    if "₹" in k:
        print(f"  {k:<22}: ₹{int(v):>12,}")
    else:
        print(f"  {k:<22}: {int(v):>12,}")

grand_tc = int(df_opt["Monthly TC ₹"].sum())
print(f"\n  ★ TOTAL MINIMUM COST    : ₹{grand_tc:,}")

# =============================================================================
# SECTION 4 — SDR TRIAL COMPARISON  (all 10 trials from Sheet 3)
# =============================================================================
print("\n" + "=" * 70)
print("SECTION 3: SDR TRIAL COMPARISON — ALL TRIALS")
print("=" * 70)

# Each trial is a 6-element list [W1..W6]
trials = {
    "T1  Level W=57 (baseline)"           : [57,55,57,57,57,57],
    "T2  Level W=55"                       : [55,55,55,55,55,55],
    "T3  Chase demand (modified)"          : [52,52,60,65,58,52],
    "T4  Level W=54"                       : [54,54,54,54,54,54],
    "T5  Moderate flex W55-60"             : [55,55,60,60,55,55],
    "T6  Full chase strategy"              : [50,55,60,70,64,54],
    "T7  Perturb period 2 +1"             : [55,56,55,55,55,55],
    "T8  Perturb period 1 +1"             : [56,55,55,55,55,55],
    "T9  Perturb period 3 +1"             : [55,55,56,55,55,55],
    "T10 SDR CONVERGED — OPTIMAL"         : [55,55,55,55,55,55],
}

# Expected decisions from the Excel file
decisions = {
    "T1  Level W=57 (baseline)"  : "BASELINE",
    "T2  Level W=55"             : "✓ ACCEPT",
    "T3  Chase demand (modified)": "✗ REJECT",
    "T4  Level W=54"             : "✗ REJECT",
    "T5  Moderate flex W55-60"   : "✗ REJECT",
    "T6  Full chase strategy"    : "✗ REJECT",
    "T7  Perturb period 2 +1"   : "✗ REJECT",
    "T8  Perturb period 1 +1"   : "✗ REJECT",
    "T9  Perturb period 3 +1"   : "✗ REJECT",
    "T10 SDR CONVERGED — OPTIMAL": "★ OPTIMAL",
}

print(f"\n{'Trial':<10}{'W1':>4}{'W2':>4}{'W3':>4}{'W4':>4}{'W5':>4}{'W6':>4}"
      f"{'Labour ₹':>12}{'Hire ₹':>10}{'Fire ₹':>10}{'Hold ₹':>10}{'Back ₹':>10}{'TOTAL TC ₹':>14}{'Decision':>14}")
print("-" * 120)

trial_results = []
for label, wplan in trials.items():
    _, tot = compute_plan(wplan, demand, I0, W0,
                          prod_rate, c_hire, c_fire, c_hold, c_back, c_labour)
    tc = int(tot["Labour Cost ₹"] + tot["Hire Cost ₹"] + tot["Fire Cost ₹"]
             + tot["Hold Cost ₹"] + tot["Back Cost ₹"])
    dec = decisions[label]
    tnum = label.split()[0]
    print(f"{tnum:<10}{wplan[0]:>4}{wplan[1]:>4}{wplan[2]:>4}{wplan[3]:>4}{wplan[4]:>4}{wplan[5]:>4}"
          f"{int(tot['Labour Cost ₹']):>12,}{int(tot['Hire Cost ₹']):>10,}{int(tot['Fire Cost ₹']):>10,}"
          f"{int(tot['Hold Cost ₹']):>10,}{int(tot['Back Cost ₹']):>10,}{tc:>14,}{dec:>14}")
    trial_results.append((tnum, tc))

print("\nSDR CONVERGENCE RULE: Algorithm stops when no single-variable")
print(f"perturbation (±1 worker in any period) reduces Total Cost below ₹{grand_tc:,}")

# =============================================================================
# SECTION 5 — COST BREAKDOWN  (Sheet 4)
# =============================================================================
print("\n" + "=" * 70)
print("SECTION 4: OPTIMAL PLAN — COMPLETE COST BREAKDOWN")
print("=" * 70)

cost_cols = ["Month", "Labour Cost ₹", "Hire Cost ₹", "Fire Cost ₹",
             "Hold Cost ₹", "Back Cost ₹", "Monthly TC ₹"]
print("\n", df_opt[cost_cols].to_string(index=False))

print("\nCOST COMPONENT ANALYSIS:")
components = [
    ("Regular Labour", int(totals_opt["Labour Cost ₹"])),
    ("Holding Cost",   int(totals_opt["Hold Cost ₹"])),
    ("Hiring Cost",    int(totals_opt["Hire Cost ₹"])),
    ("Backorder Cost", int(totals_opt["Back Cost ₹"])),
    ("Firing Cost",    int(totals_opt["Fire Cost ₹"])),
]
total_check = sum(v for _, v in components)
print(f"\n  {'Component':<22} {'Amount (₹)':>14} {'% of Total TC':>16}  Remarks")
print("  " + "-" * 80)
for name, amt in components:
    pct = amt / total_check * 100
    print(f"  {name:<22} ₹{amt:>12,}   {pct:>14.2f}%")
print("  " + "-" * 80)
print(f"  {'TOTAL':<22} ₹{total_check:>12,}   {'100.00%':>15}")

# =============================================================================
# SECTION 6 — INVENTORY TRACE  (Sheet 5)
# =============================================================================
print("\n" + "=" * 70)
print("SECTION 5: INVENTORY BALANCE TRACE — OPTIMAL PLAN (W=55)")
print("=" * 70)

print(f"\n{'Month':<12}{'Iₜ₋₁':>8}{'Prod Pt':>10}{'Demand Dt':>12}{'Inv It':>10}  Status           Cost Detail")
print("-" * 80)

inv_prev = I0
W_prev   = W0
for t in range(6):
    Wt  = 55
    Pt  = Wt * prod_rate
    Dt  = demand[t]
    It  = inv_prev + Pt - Dt
    hold_c = max(0,  It) * c_hold
    back_c = max(0, -It) * c_back
    if It > 0:
        status  = "Holding"
        detail  = f"₹{hold_c:,} holding"
    elif It < 0:
        status  = "BACKORDER ⚠"
        detail  = f"₹{back_c:,} backorder"
    else:
        status  = "Zero"
        detail  = "₹0"
    print(f"  {months[t]:<10}{inv_prev:>8,}{Pt:>10,}{Dt:>12,}{It:>10,}  {status:<15}  {detail}")
    inv_prev = It
    W_prev   = Wt

print("\nFormula: It = Iₜ₋₁ + Pt − Dt")
print(f"  Holding  cost = MAX(0,  It) × ₹{c_hold}  |  Backorder cost = MAX(0, −It) × ₹{c_back}")
print(f"  I₀ = {I0} units (given)")

# =============================================================================
# SECTION 7 — FINAL SUMMARY
# =============================================================================
print("\n" + "=" * 70)
print("FINAL SUMMARY — SDR OPTIMAL SOLUTION")
print("=" * 70)
print(f"\n  Optimal Workforce : W* = 55 workers  (Level Strategy)")
print(f"  Planning Horizon  : 6 months  (Jan – Jun)")
print()
print(f"  Labour Cost       : ₹{int(totals_opt['Labour Cost ₹']):>12,}  (98.18% of total)")
print(f"  Holding Cost      : ₹{int(totals_opt['Hold Cost ₹']):>12,}  (inventory Jan–Apr)")
print(f"  Hiring Cost       : ₹{int(totals_opt['Hire Cost ₹']):>12,}  (5 workers in Jan)")
print(f"  Backorder Cost    : ₹{int(totals_opt['Back Cost ₹']):>12,}  (50 units short in May)")
print(f"  Firing Cost       : ₹{int(totals_opt['Fire Cost ₹']):>12,}  (zero firings)")
print(f"  {'─'*42}")
print(f"  TOTAL MINIMUM COST: ₹{grand_tc:>12,}")
print()
print("  VERIFICATION:")
print(f"    ₹3,960,000 + ₹25,000 + ₹0 + ₹46,000 + ₹2,500 = ₹{3960000+25000+0+46000+2500:,}")
print(f"    Excel SUM(Monthly TC) = ₹{grand_tc:,}  |  MATCH CONFIRMED ✓")

Running in Google Colab — uploading file …


Saving SDR_Aggregate_Planning.xlsx to SDR_Aggregate_Planning.xlsx
SECTION 1: READING PROBLEM DATA FROM EXCEL
Sheets found: ['1_Problem_Data', '2_SDR_Optimal_Plan', '3_SDR_Trial_Comparison', '4_Cost_Breakdown', '5_Inventory_Trace']

GIVEN DATA:
  Months  : ['January', 'February', 'March', 'April', 'May', 'June']
  Demand  : [1200, 1500, 1800, 2100, 1900, 1600]
  Total D : 10100 units  |  Average D : 1683.33 units/month

COST & OPERATIONAL PARAMETERS:
  Production rate  (p)       : 30 units/worker/month
  Initial workforce (W0)     : 50 workers
  Initial inventory (I0)     : 200 units
  Hiring cost  (c_hire)      : ₹5,000/worker
  Firing cost  (c_fire)      : ₹8,000/worker
  Holding cost (c_hold)      : ₹20/unit/month
  Backorder cost (c_back)    : ₹50/unit/month
  Labour cost  (c_labour)    : ₹12,000/worker/month

SECTION 2: SDR OPTIMAL PLAN  (W* = 55 workers — Level Strategy)

MONTH-BY-MONTH DETAIL:
   Month  Workforce Wt  Production Pt  Demand Dt  ΔW  Hire Ht  Fire Ft  Inventory It  L